In [ ]:
!uv pip install lightgbm
!uv pip install --upgrade mlflow 
!uv pip install cloudpickle psutil
!uv pip install tensorflow
!uv pip install tensorboard

In [27]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

# 실험 설정
mlflow.set_experiment("NASA_Engine_RUL_7Models_with_Val")

<Experiment: artifact_location='file:c:/python_course/4project_cmapss/SH/mlruns/1', creation_time=1776515554182, experiment_id='1', last_update_time=1776515554182, lifecycle_stage='active', name='NASA_Engine_RUL_7Models_with_Val', tags={}, trace_location=None, workspace='default'>

In [28]:
# 1. 파일 경로 및 설정 (이미 뽑아놓은 파일명과 일치해야 함)
S_COUNT = 14
CAP = 125
SIGMA = 2
SCALER = "minmax"
DATA_DIR = 'preprocessed'
file_base = f"FD001_S{S_COUNT}_{CAP}_a{SIGMA}_{SCALER}"

# 2. 데이터 로드
tr_p = pd.read_csv(os.path.join(DATA_DIR, f"train_{file_base}.csv"))
te_p = pd.read_csv(os.path.join(DATA_DIR, f"test_{file_base}.csv"))
rul_p = pd.read_csv(os.path.join(DATA_DIR, f"RUL_FD001_{CAP}.csv"))

# 3. [핵심] USEFUL_SENSORS를 CSV 컬럼에서 자동으로 추출
# unit_nr, time_cycles, RUL을 제외한 나머지가 모두 센서입니다.
USEFUL_SENSORS = [col for col in tr_p.columns if col not in ['unit_nr', 'time_cycles', 'RUL']]

print(f"✅ 데이터 로드 완료! 사용된 센서: {len(USEFUL_SENSORS)}개")
print(f"학습 데이터: {tr_p.shape}, 테스트 데이터: {te_p.shape}, 테스트 정답지: {rul_p.shape}")

✅ 데이터 로드 완료! 사용된 센서: 14개
학습 데이터: (20631, 17), 테스트 데이터: (13096, 16), 테스트 정답지: (100, 1)


In [ ]:
# 엔진 단위로 Train/Valid 분할 (8:2)
def split_units(df, test_size=0.2):
    units = df['unit_nr'].unique()
    train_units, val_units = train_test_split(units, test_size=test_size, random_state=42)
    return df[df['unit_nr'].isin(train_units)], df[df['unit_nr'].isin(val_units)]

def prepare_ml_features(df, sensors, window_size=30):
    df_ml = df.copy()
    for s in sensors:
        df_ml[f'{s}_mean'] = df_ml.groupby('unit_nr')[s].transform(lambda x: x.rolling(window_size).mean())
        df_ml[f'{s}_std'] = df_ml.groupby('unit_nr')[s].transform(lambda x: x.rolling(window_size).std())
    return df_ml.dropna().reset_index(drop=True)

# 데이터 준비
full_train_ml = prepare_ml_features(tr_p, USEFUL_SENSORS, window_size=30)
train_df_ml, val_df_ml = split_units(full_train_ml)

X_train_ml = train_df_ml.drop(columns=['unit_nr', 'RUL'])
y_train_ml = train_df_ml['RUL']
X_val_ml = val_df_ml.drop(columns=['unit_nr', 'RUL'])
y_val_ml = val_df_ml['RUL']

def run_ml_experiment(model, model_name, X_train, y_train, X_val, y_val):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        
        # 검증 점수 계산
        val_preds = model.predict(X_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("window_size", 30)
        mlflow.log_metric("val_rmse", val_rmse) # 검증 RMSE 기록
        # mlflow.sklearn.log_model(model, "model")
        print(f"✅ ML {model_name} 완료 (Val RMSE: {val_rmse:.4f})")

# ML 실행
ml_models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_jobs=-1, random_state=42)
}

for name, model in ml_models.items():
    run_ml_experiment(model, name, X_train_ml, y_train_ml, X_val_ml, y_val_ml)

2026/04/18 22:10:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:10:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 22:10:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ ML LinearRegression 완료 (Val RMSE: 15.1124)


2026/04/18 22:11:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:11:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 22:11:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/18 22:11:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:11:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, wh

✅ ML RandomForest 완료 (Val RMSE: 12.1197)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001500 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10965
[LightGBM] [Info] Number of data points in the train set: 14241, number of used features: 43
[LightGBM] [Info] Start training from score 80.812373


2026/04/18 22:11:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


✅ ML LightGBM 완료 (Val RMSE: 12.0579)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def create_dl_sequences(df, sensors, window_size=30):
    feature_list, label_list = [], []
    for unit in df['unit_nr'].unique():
        unit_df = df[df['unit_nr'] == unit]
        if len(unit_df) >= window_size:
            data = unit_df[sensors].values
            labels = unit_df['RUL'].values
            for i in range(len(unit_df) - window_size + 1):
                feature_list.append(data[i:i+window_size, :])
                label_list.append(labels[i+window_size-1])
    return np.array(feature_list), np.array(label_list)

# 데이터 분할 및 시퀀스 생성
train_df_dl, val_df_dl = split_units(tr_p)
X_train_dl, y_train_dl = create_dl_sequences(train_df_dl, USEFUL_SENSORS, 30)
X_val_dl, y_val_dl = create_dl_sequences(val_df_dl, USEFUL_SENSORS, 30)

def run_dl_experiment(model_builder, model_name, X_train, y_train, X_val, y_val):
    with mlflow.start_run(run_name=model_name):
        # mlflow.tensorflow.autolog() # 학습 중 Val Loss 자동 기록
        
        model = model_builder()
        model.compile(optimizer='adam', loss='mse')
        
        # 밸리데이션 데이터를 직접 전달
        model.fit(X_train, y_train, 
                  validation_data=(X_val, y_val), 
                  epochs=10, batch_size=64, verbose=0)
        
        val_preds = model.predict(X_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        mlflow.log_metric("final_val_rmse", val_rmse)
        print(f"✅ DL {model_name} 완료 (Val RMSE: {val_rmse:.4f})")

# 모델 빌더들
def build_cnn():
    model = models.Sequential([
        layers.Conv1D(64, 3, activation='relu', input_shape=(30, len(USEFUL_SENSORS))),
        layers.GlobalMaxPooling1D(), layers.Dense(32, activation='relu'), layers.Dense(1)
    ])
    return model

def build_lstm():
    model = models.Sequential([
        layers.LSTM(64, input_shape=(30, len(USEFUL_SENSORS))),
        layers.Dense(32, activation='relu'), layers.Dense(1)
    ])
    return model

def build_transformer():
    inputs = layers.Input(shape=(30, len(USEFUL_SENSORS)))
    att = layers.MultiHeadAttention(num_heads=4, key_dim=len(USEFUL_SENSORS))(inputs, inputs)
    gap = layers.GlobalAveragePooling1D()(att)
    outputs = layers.Dense(1)(layers.Dense(32, activation='relu')(gap))
    return models.Model(inputs, outputs)

def build_cnn_trans():
    inputs = layers.Input(shape=(30, len(USEFUL_SENSORS)))
    x = layers.Conv1D(64, 3, activation='relu')(inputs)
    att = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    gap = layers.GlobalAveragePooling1D()(att)
    outputs = layers.Dense(1)(gap)
    return models.Model(inputs, outputs)

# DL 실행
dl_models = [(build_cnn, "1D-CNN"), (build_lstm, "LSTM"), 
             (build_transformer, "Transformer"), (build_cnn_trans, "CNN_Transformer")]

for builder, name in dl_models:
    run_dl_experiment(builder, name, X_train_dl, y_train_dl, X_val_dl, y_val_dl)

c:\python_course\4project_cmapss\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


2026/04/18 22:21:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:21:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 742us/step
✅ DL 1D-CNN 완료 (Val RMSE: 18.2084)


c:\python_course\4project_cmapss\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

2026/04/18 22:22:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:22:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ DL Transformer 완료 (Val RMSE: 18.0615)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


2026/04/18 22:22:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 22:22:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ DL CNN_Transformer 완료 (Val RMSE: 17.0925)
